# Tool catalog: file, playback, and clipboard

Run these cells in order in a disposable JupyterLab project with its live kernel. The blue PNG and five-second WAV are generated inputs. Click the file picker, Play, Copy, or paste control when it appears; no browser can perform those user actions silently. Each call returns an immediate receipt, so inspect it on a later turn after the media row completes. The AI questions ask the selected ChatGPT connection to make real, scoped tool calls. Do not run all cells at once.


## Disposable exact media references

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib, math, struct, wave
from PIL import Image
from IPython.display import display, Audio

server_root = Path.cwd().resolve()
nonce = uuid4().hex
image_path = server_root / f'playback-{nonce}.png'
tone_path = server_root / f'playback-{nonce}.wav'
Image.new('RGB', (2, 2), (36, 112, 195)).save(image_path)
with wave.open(str(tone_path), 'wb') as output:
    output.setnchannels(1); output.setsampwidth(2); output.setframerate(16000)
    output.writeframes(b''.join(struct.pack('<h', int(9000 * math.sin(2 * math.pi * 440 * i / 16000))) for i in range(5 * 16000)))
image_ref = {'path': image_path.name, 'sha256': hashlib.sha256(image_path.read_bytes()).hexdigest()}
tone_ref = {'path': tone_path.name, 'sha256': hashlib.sha256(tone_path.read_bytes()).hexdigest()}
print('Generated five-second 440 Hz WAV:', tone_path.name)
print('Generated blue PNG:', image_path.name)
display(Image.open(image_path).resize((80, 80)))
display(Audio(filename=str(tone_path)))


## choose_file

In [ ]:
from nbinlineai.tools import choose_file
selected = choose_file(accept='image/png')
selected  # In the visible chooser select only the generated playback-*.png sample.


In [ ]:
print('Chosen file:', selected.status, selected.error)
if selected.status == 'completed':
    print('Imported image:', {key: selected.media.get(key) for key in ('media_id', 'mime_type', 'sha256')})
    display(selected.result.resize((80, 80)))


Use &`choose_file` with `accept='image/png'` so I can select this notebook's generated blue PNG in the visible picker. Report its actual MIME type and exact SHA-256 after I select it, then use &`release_media` on that imported managed media ID. Do not choose any personal file.


## open_media

In [ ]:
from nbinlineai.tools import open_media
opened = open_media(tone_ref)
opened  # The owned preview uses the exact WAV bytes and hash.


In [ ]:
print('Open preview:', opened.status, opened.error)
if opened.status == 'completed':
    preview_id = opened.result['preview_id']
    print('Preview ID:', preview_id)


Use &`open_media` with the exact five-second WAV reference $`tone_ref`. Report the actual preview ID and media type after it opens, then use &`close_media` on that new preview to avoid leaving a second player.


## play_media

In [ ]:
from nbinlineai.tools import play_media
assert opened.status == 'completed', 'Wait for the preview, then rerun this cell.'
playing = play_media(opened.result['preview_id'])
playing  # Click Play in the visible panel if the browser requires activation.


In [ ]:
print('Play:', playing.status, playing.result, playing.error)


Use &`play_media` for preview ID $`preview_id`. Wait for any visible Play action I must click, then report the actual playback state.


## pause_media

In [ ]:
from nbinlineai.tools import pause_media
paused = pause_media(opened.result['preview_id'])
paused


In [ ]:
print('Pause:', paused.status, paused.result, paused.error)


Use &`pause_media` for preview ID $`preview_id`; report the actual paused state and the ID.


## seek_media

In [ ]:
from nbinlineai.tools import seek_media
seeked = seek_media(opened.result['preview_id'], 2.5)
seeked


In [ ]:
print('Seek:', seeked.status, seeked.result, seeked.error)


Use &`seek_media` for preview ID $`preview_id` at 2.5 seconds of the generated five-second tone. Report the actual position, which may differ slightly from the request.


## set_media_volume

In [ ]:
from nbinlineai.tools import set_media_volume
volume = set_media_volume(opened.result['preview_id'], 0.25)
volume


In [ ]:
print('Volume:', volume.status, volume.result, volume.error)


Use &`set_media_volume` for preview ID $`preview_id` with volume 0.25. Report the real result or a browser restriction without claiming volume changed if it did not.


## close_media

In [ ]:
from nbinlineai.tools import close_media
closed = close_media(opened.result['preview_id'])
closed  # Closing the preview keeps tone_path intact.


In [ ]:
print('Close:', closed.status, closed.result, closed.error)
print('Original five-second WAV remains:', tone_path.exists())


Use &`open_media` on $`tone_ref` to make a fresh preview, then use &`close_media` on that new preview ID. Report both actual states and explain that closing it leaves the generated WAV file intact.


## copy_text

In [ ]:
from nbinlineai.tools import copy_text
copied = copy_text('nbinlineai disposable playback sample')
copied  # Click the visible Copy button or use the browser copy shortcut.


In [ ]:
print('Copy:', copied.status, copied.result, copied.error)


Use &`copy_text` with the literal `nbinlineai disposable playback sample`. Ask me to click the visible Copy control if needed, then report the actual completion.


## paste_content

In [ ]:
from nbinlineai.tools import paste_content
pasted = paste_content(accept='text')
pasted  # Click the visible paste box and paste the sample text.


In [ ]:
print('Paste:', pasted.status, pasted.error)
if pasted.status == 'completed':
    print('Pasted text:', pasted.result)


Use &`paste_content` with `accept='text'`. Ask me to paste the disposable sample into the visible paste box, then quote only the returned text and whether it was truncated.


In [ ]:
assert tone_path.exists() and image_path.exists()
tone_path.unlink()  # Delete only these two generated sample files.
image_path.unlink()
print('Disposable playback samples removed.')
